# Analysis of Variance (one-way ANOVA)

Companion to [the ANOVA note](../../notes/regression/analysis_of_variance.md). One-way ANOVA tests $H_0:\mu_1=\dots=\mu_k$ by splitting total variability into between-group ($SST$) and within-group ($SSE$) parts and comparing mean squares with an F-statistic. Here we simulate apple weights from three farms and walk through the test.

In [1]:
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)

## Simulated data
Three farms with true mean weights 148, 150 and 152 g and common standard deviation 10 g, 50 apples each. The true means differ by a few grams, but the noise is large, so the test may not detect it.

In [2]:
farms = {name: rng.normal(mu, 10, 50) for name, mu in [('A',148),('B',150),('C',152)]}
groups = list(farms.values())
for n, g in farms.items():
    print(f'Farm {n}: mean={g.mean():.2f}, sd={g.std(ddof=1):.2f}')

Farm A: mean=148.91, sd=7.68
Farm B: mean=148.08, sd=7.67
Farm C: mean=151.49, sd=10.04


## The test
`scipy.stats.f_oneway` returns the F-statistic and its p-value from an $F_{k-1,\,N-k}$ distribution.

In [3]:
f_stat, p_value = stats.f_oneway(*groups)
print(f'F = {f_stat:.2f}, p = {p_value:.4f}')
print('Reject H0' if p_value < 0.05 else 'Fail to reject H0 at alpha = 0.05')

F = 2.16, p = 0.1186
Fail to reject H0 at alpha = 0.05


## ANOVA table by hand
We recompute $SST=\sum n_j(\bar y_j-\bar y)^2$, $SSE=\sum\sum (y_{ij}-\bar y_j)^2$, and check $SSTotal=SST+SSE$ and that the F-statistic matches scipy.

In [4]:
y = np.concatenate(groups)
k, N = len(groups), y.size
grand = y.mean()
sst = sum(g.size*(g.mean()-grand)**2 for g in groups)
sse = sum(((g-g.mean())**2).sum() for g in groups)
sstotal = ((y-grand)**2).sum()
mst, mse = sst/(k-1), sse/(N-k)
print(f'SST={sst:.1f}  SSE={sse:.1f}  SST+SSE={sst+sse:.1f}  SSTotal={sstotal:.1f}')
print(f'MST={mst:.1f} (df={k-1})  MSE={mse:.1f} (df={N-k})  F={mst/mse:.2f}')
print('p =', round(stats.f.sf(mst/mse, k-1, N-k), 4))

SST=315.3  SSE=10711.5  SST+SSE=11026.7  SSTotal=11026.7
MST=157.6 (df=2)  MSE=72.9 (df=147)  F=2.16
p = 0.1186


## Visualizing the groups
Boxplots show the large overlap between farms compared with the small differences in means, which is why $F$ is small.

In [5]:
plt.figure(figsize=(7,4))
plt.boxplot(groups, patch_artist=True, boxprops=dict(facecolor='skyblue'))
plt.xticks([1,2,3], ['Farm A','Farm B','Farm C'])
plt.ylabel('Weight (g)')
plt.title('Apple weights by farm')
plt.grid(True, axis='y', linestyle='--', alpha=0.6)
plt.show()

/tmp/ipykernel_78566/3020537359.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Checking assumptions and post-hoc tests
Levene's test checks equal variances and Shapiro-Wilk checks residual normality. If the F-test had rejected, Tukey's HSD would identify which pairs differ.

In [6]:
print('Levene p =', round(stats.levene(*groups).pvalue, 3))
resid = np.concatenate([g-g.mean() for g in groups])
print('Shapiro-Wilk p =', round(stats.shapiro(resid).pvalue, 3))
print(stats.tukey_hsd(*groups))

Levene p = 0.18
Shapiro-Wilk p = 0.435
Pairwise Group Comparisons (95.0% Confidence Interval)
Comparison  Statistic  p-value  Lower CI  Upper CI
 (0 - 1)      0.830     0.878    -3.213     4.872
 (0 - 2)     -2.576     0.290    -6.618     1.467
 (1 - 0)     -0.830     0.878    -4.872     3.213
 (1 - 2)     -3.405     0.117    -7.447     0.637
 (2 - 0)      2.576     0.290    -1.467     6.618
 (2 - 1)      3.405     0.117    -0.637     7.447

